# Feature Engineering
Feature engineering is the process of using domain knowledge to create new features from existing data. It involves transforming raw data into a format that is more suitable for machine learning algorithms, which can lead to improved model performance.
- In this notebook of the house price predictor project, I will be performing feature engineering on the dataset to create new features that can help improve the predictive power of the model. This may include creating interaction terms, polynomial features, or aggregating existing features to capture more complex relationships in the data.

## Data loading

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from urllib.parse import quote_plus
from google.colab import userdata

In [ ]:
DB_USER = userdata.get("SUPABASE_USER")
DB_PASSWORD = quote_plus(userdata.get("SUPABASE_PASSWORD"))
DB_HOST = userdata.get("SUPABASE_HOST")
DB_PORT = userdata.get("SUPABASE_PORT")
DB_NAME = userdata.get("SUPABASE_NAME")

# 2. Build SQLAlchemy connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
with engine.connect() as conn:
    df = pd.read_sql(
        text("SELECT * FROM housing_data"),
        conn
    )

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=["Order", "PID"])

In [ ]:
df.head()

## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

## Features classification

**1. Numerical features**

In [ ]:
# define numerical features
numerical_features_all = df.select_dtypes(include=[np.number]).columns.tolist()

In [ ]:
# Discrete features
discrete_num_features = ['MS_SubClass', 'Overall_Qual', 'Overall_Cond', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Cars']
discrete_num_features

In [ ]:
# Continuous features
continuous_num_features = list(set(numerical_features_all) - set(discrete_num_features))
continuous_num_features

In [ ]:
num_feats_int = list(set(numerical_features_all) - set(discrete_num_features)) + ['SalePrice']

**2. Categorical features**

In [ ]:
# define categorical features
categorical_features = df.select_dtypes(include=[object]).columns.tolist()

In [ ]:
# Add discrete features to categorical features list
categorical_features_all = list(set(categorical_features + discrete_num_features))

In [ ]:
categorical_features_all

**Change data types of `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [ ]:
for col in categorical_features_all:
    df[col] = df[col].astype('category')

In [ ]:
df[categorical_features_all].info()

**3. Temporal features**

In [ ]:
date_year = ['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', ]
date_month = ['Mo_Sold']

for col in date_year:
    df[col] = pd.to_datetime(df[col], format='%Y').dt.year

for col in date_month:
    df[col] = pd.to_datetime(df[col], format='%m').dt.month

In [ ]:
# Temporal features
temporal_features = date_year + date_month
temporal_features

**Summarize**

In [ ]:

# print columns
print(f'{len(numerical_features_all)} numerical features: {numerical_features_all}')
print(f'\n{len(categorical_features)} categorical features: {categorical_features}')
print(f'\n{len(temporal_features)} temporal features: {temporal_features}')

In [ ]:
# total
print(f'Total features: {len(numerical_features_all) + len(categorical_features)}')

## 1. Outlier Removal/Resolving

Important concepts in outlier detection! Here's a breakdown:

*   **Interquartile Range (IQR):** The IQR is a measure of statistical dispersion, representing the range of the middle 50% of your data. It's calculated as the difference between the third quartile (Q3) and the first quartile (Q1).

    *   **Q1 (First Quartile):** The value below which 25% of the data falls.
    *   **Q3 (Third Quartile):** The value below which 75% of the data falls.
    *   `IQR = Q3 - Q1`

*   **Outlier Thresholds (Lower Bound and Upper Bound):** These are calculated boundaries used to identify potential outliers in a dataset. They are derived using the IQR.

    *   **Lower Bound:** `Q1 - 1.5 * IQR`
    *   **Upper Bound:** `Q3 + 1.5 * IQR`

*   **Outlier vs. IQR:**
    *   The **IQR** itself is a measure of the spread of the central portion of the data, robust to extreme values.
    *   **Outliers** are data points that fall outside the calculated lower and upper bounds. The 1.5 multiplier in the threshold calculation is a commonly used convention (often referred to as the "1.5 IQR rule") to define what constitutes an outlier relative to the bulk of the data's spread. Any data point less than the lower bound or greater than the upper bound is considered an outlier.

In [ ]:
num_feats_int

In [ ]:
# Calculate outlier thresholds
def outlier_thresholds(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    #outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return lower_bound, upper_bound

In [ ]:
# detect outliers
def detect_outliers(df, column):
    if pd.api.types.is_numeric_dtype(df[column]):
        low, up = outlier_thresholds(df, column)
        return (df[column] > up) | (df[column] < low)
    else:
        return pd.Series(False, index=df.index)

In [ ]:
outliers_df = pd.DataFrame(index=df.index)
for col in numerical_features_all:
    outliers_df[col] = detect_outliers(df, col)

# Display the count of outliers per column
print("Number of outliers per numerical column:")
print(outliers_df.sum())

In [ ]:
len(numerical_features_all)

In [ ]:
len(discrete_num_features)

In [ ]:
len(num_feats_int)

> $26 = 37 - 12 + 1$

In [ ]:
display(outliers_df.head())

In [ ]:
# Function to check all numeric columns for outliers
def inspect_outliers(df, col_list):
    results = {}
    for col in col_list:
        results[col] = detect_outliers(df, col).any() if pd.api.types.is_numeric_dtype(df[col]) else False
    return results

In [ ]:
# Function to display columns with outlier information
def display_outlier_info(df, col_list):
    print("Outlier thresholds for the numeric column")
    for col in col_list:
        low, up = outlier_thresholds(df, col)
        print(f"{col} : Lower Bound: {low}, Upper Bound: {up}", end="\n")
    print(end="\n\n")
    outliers_absent = []
    outliers_present = []
    for col, value in inspect_outliers(df, col_list).items():
        if value:
            outliers_present.append(col)
        else:
            outliers_absent.append(col)
    print("Columns with outliers")
    print(outliers_present)
    print(f"column count: {len(outliers_present)}", end="\n\n")
    print("Columns with no outliers")
    print(outliers_absent)
    print(f"column count: {len(outliers_absent)}", end="\n\n")

    return outliers_present, outliers_absent

In [ ]:
# Resolve Outliers
def threshold_resolution(df, column):
    low, up = outlier_thresholds(df, column)
    df.loc[df[column] < low, column] = low
    df.loc[df[column] > up, column] = up
    return low, up

In [ ]:
current_numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
outliers_present, outliers_absent = display_outlier_info(df, current_numerical_columns)

**Now remove outliers**

In [ ]:
for col in outliers_present:
    if col not in ['SalePrice']:
        threshold_resolution(df, col)

In [ ]:
outliers_present, outliers_absent = display_outlier_info(df, current_numerical_columns)

> **Now no columns with outliers except `SalePrice`**

## 2. Train-Test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
y_train.head()

In [ ]:
y_test.head()

## 3. Missing Value Handling (Imputation)

In [ ]:
count = df.isnull().sum().sort_values(ascending=False)
percentage = ((df.isnull().sum() / df.isnull().count()) * 100).sort_values(ascending=False)
null_values = pd.concat([count, percentage], axis=1, keys=['Count', 'Percentage'])

In [ ]:
null_values.head()

In [ ]:
df.columns.to_list()

**Visualize missing values**

In [ ]:
fig = px.bar(x=null_values.index, y=null_values['Percentage'],
             labels={'x': 'Features', 'y': 'Percent of missing values'},
             title='Percent missing data by feature')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**1. Structural Absence (Replace `NaN` with `None` or `0`)**

These columns use missing values to denote that the feature does not exist on the property.

- **`structural_categorical` (Fill with `None`)**

In [ ]:
structural_categorical = [
    'Alley', 'Mas_Vnr_Type', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure',
    'BsmtFin_Type_1', 'BsmtFin_Type_2', 'Fireplace_Qu', 'Garage_Type',
    'Garage_Finish', 'Garage_Qual', 'Garage_Cond', 'Pool_QC', 'Fence', 'Misc_Feature'
]

```
df.Alley=["No Alley" if x is np.nan else x for x in df.Alley]
df.Mas_Vnr_Type=["Missing" if x is np.nan else x for x in df.Mas_Vnr_Type]
df.Bsmt_Qual=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Qual]
df.Bsmt_Cond=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Cond]
df.Bsmt_Exposure=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Exposure]
df.BsmtFin_Type_1=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_1]
df.BsmtFin_Type_2=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_2]
df.Fireplace_Qu=["No Firepl" if x is np.nan else x for x in df.Fireplace_Qu]
df.Garage_Type=["No Grg" if x is np.nan else x for x in df.Garage_Type]
df.Garage_Finish=["No Grg" if x is np.nan else x for x in df.Garage_Finish]
df.Garage_Qual=["No Grg" if x is np.nan else x for x in df.Garage_Qual]
df.Garage_Cond=["No Grg" if x is np.nan else x for x in df.Garage_Cond]
df.Pool_QC=["No Pool" if x is np.nan else x for x in df.Pool_QC]
df.Fence=["No Fence" if x is np.nan else x for x in df.Fence]
df.Misc_Feature=["None" if x is np.nan else x for x in df.Misc_Feature]
```

```py
df['Alley'] = df['Alley'].fillna('No Alley')
df['Mas_Vnr_Type'] = df['Mas_Vnr_Type'].fillna('Missing')
df['Bsmt_Qual'] = df['Bsmt_Qual'].fillna('No Bsmnt')
df['Bsmt_Cond'] = df['Bsmt_Cond'].fillna('No Bsmnt')
df['Bsmt_Exposure'] = df['Bsmt_Exposure'].fillna('No Bsmnt')
df['BsmtFin_Type_1'] = df['BsmtFin_Type_1'].fillna('No Bsmnt')
df['BsmtFin_Type_2'] = df['BsmtFin_Type_2'].fillna('No Bsmnt')
df['Fireplace_Qu'] = df['Fireplace_Qu'].fillna('No Firepl')
df['Garage_Type'] = df['Garage_Type'].fillna('No Grg')
df['Garage_Finish'] = df['Garage_Finish'].fillna('No Grg')
df['Garage_Qual'] = df['Garage_Qual'].fillna('No Grg')
df['Garage_Cond'] = df['Garage_Cond'].fillna('No Grg')
df['Pool_QC'] = df['Pool_QC'].fillna('No Pool')
df['Fence'] = df['Fence'].fillna('No Fence')
df['Misc_Feature'] = df['Misc_Feature'].fillna('None')
```

In [ ]:
fill_values = {
    'Alley': 'No Alley',
    'Mas_Vnr_Type': 'Missing',
    'Bsmt_Qual': 'No Bsmnt',
    'Bsmt_Cond': 'No Bsmnt',
    'Bsmt_Exposure': 'No Bsmnt',
    'BsmtFin_Type_1': 'No Bsmnt',
    'BsmtFin_Type_2': 'No Bsmnt',
    'Fireplace_Qu': 'No Firepl',
    'Garage_Type': 'No Grg',
    'Garage_Finish': 'No Grg',
    'Garage_Qual': 'No Grg',
    'Garage_Cond': 'No Grg',
    'Pool_QC': 'No Pool',
    'Fence': 'No Fence',
    'Misc_Feature': 'None'
}

for col, fill_value in fill_values.items():
    if fill_value not in df[col].cat.categories:
        df[col] = df[col].cat.add_categories(fill_value)
    df[col] = df[col].fillna(fill_value)

In [ ]:
df[structural_categorical].isnull().sum()

- **`structural_numerical` (Fill with `0`)**

In [ ]:
structural_numerical = [
    'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF',
    'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Garage_Cars', 'Garage_Area', 'Mas_Vnr_Area'
]

In [ ]:
for col in structural_numerical:
    df[col] = df[col].fillna(0)

In [ ]:
df[structural_numerical].isnull().sum()

**Checking**

In [ ]:
df[structural_numerical + structural_categorical].isnull().sum().sum()

In [ ]:
len(structural_numerical + structural_categorical)

In [ ]:
features_with_missing_values = list(set(df.columns.to_list()) - set(structural_numerical + structural_categorical))
print(len(features_with_missing_values))
features_with_missing_values

> Out of $80$ total features, $23$ features with missing values are handled considering _structuring absence_. Remaining $57$ will be handled by _statistical imputation_.

In [ ]:
df.head()

**2. Statistical Imputation (True Missingness/Clerical Errors)**

These features are physically required for a house to exist. Missing data here represents a processing omission or clerical typo.

- **`spatial_numerical` (Impute using the median of their specific `Neighborhood`)**

In [ ]:
spatial_numerical = ['Lot_Frontage']

In [ ]:
df[spatial_numerical].isnull().sum()

In [ ]:
df['Lot_Frontage'].interpolate(method='linear', axis=0, inplace=True)

In [ ]:
df[spatial_numerical].isnull().sum()

- **`mode_categorical` (Impute using the column's global mode/most frequent item)**

In [ ]:
mode_categorical = [
    'Utilities', 'Exterior_1st', 'Exterior_2nd', 'Electrical',
    'Kitchen_Qual', 'Functional', 'Sale_Type'
]

In [ ]:
df[mode_categorical].isnull().sum()

In [ ]:
df[mode_categorical].head()

In [ ]:
for col in mode_categorical:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

- **`special_case_temporal` (Impute missing indices dynamically with that row's `Year_Built` value)**

In [ ]:
special_case_temporal = ['Garage_Yr_Blt']

In [ ]:
df[temporal_features].isnull().sum()

In [ ]:
df[temporal_features].head()

In [ ]:
df['Garage_Yr_Blt'] = df['Garage_Yr_Blt'].fillna(df['Year_Built'])

In [ ]:
df[temporal_features].isnull().sum()

**Final check for existence of null values**

In [ ]:
# Null values left?
print(df.isnull().values.any())
if df.isnull().values.any():
  print("Yes, there exist null values!")
  null_counts = df.isnull().sum()
  null_counts = null_counts[null_counts > 0]
  print(null_counts)
else:
  print("No null values")

> **Final dataset with no missing values**

In [ ]:
df.head()

In [ ]:
df.info()

## 4. Feature Creation / Engineering

In [ ]:
import numpy as np
# New Features for Housing Dataset

# Total square footage
df['TotalSF'] = df['1st_Flr_SF'] + df['2nd_Flr_SF'] + df['Total_Bsmt_SF']

# Age of the house at time of sale
df['Age'] = df['Yr_Sold'] - df['Year_Built']

# Ensure Half_Bath and Bsmt_Half_Bath are numeric for calculations
df['Half_Bath'] = df['Half_Bath'].astype(int)
df['Bsmt_Half_Bath'] = df['Bsmt_Half_Bath'].astype(int)
df['Full_Bath'] = df['Full_Bath'].astype(int)
df['Bsmt_Full_Bath'] = df['Bsmt_Full_Bath'].astype(int)

# Total number of bathrooms
df['TotalBathrooms'] = (df['Full_Bath'] + (0.5 * df['Half_Bath']) +
                        df['Bsmt_Full_Bath'] + (0.5 * df['Bsmt_Half_Bath'])).astype('category')

# Boolean indicator for pool presence
df['HasPool'] = df['Pool_Area'].apply(lambda x: 1 if x > 0 else 0).astype('category')

# Total porch area
df['TotalPorchSF'] = df['Open_Porch_SF'] + df['Enclosed_Porch'] + df['3Ssn_Porch'] + df['Screen_Porch']

# Ensure Garage_Cars is numeric for calculations
df['Garage_Cars'] = df['Garage_Cars'].astype(int)

# Total parking capacity, considering paved drive
df['TotalParking'] = (df['Garage_Cars'] + df['Paved_Drive'].apply(lambda x:1 if x == 'Y' else 0)).astype('category')
# Boolean indicator for remodeling
df['Remodeled'] = (df['Year_Remod/Add'] != df['Year_Built']).astype(int).astype('category')
# Years since last remodel at time of sale
df['YearsSinceRemodel'] = df['Yr_Sold'] - df['Year_Remod/Add'].astype(int)

# Total number of rooms
df['TotalRooms'] = df['TotRms_AbvGrd'].astype('category') # TotRms_AbvGrd is already category

# Ratio of living area to lot area
df['LotRatio'] = df['Gr_Liv_Area'] / df['Lot_Area']

# Sale price per square foot (if SalePrice is available)
if 'SalePrice' in df.columns:
    df['PricePerSqFt'] = df['SalePrice'] / df['TotalSF']

# Categorical age group of the house
def age_group(age):
    if age <= 10:
        return 'New'
    elif age <= 30:
        return 'Modern'
    elif age <= 50:
        return 'Mature'
    else:
        return 'Old'

df['AgeGroup'] = df['Age'].apply(age_group).astype('category')

# Total outdoor living space
df['TotalOutdoorSF'] = df['Wood_Deck_SF'] + df['TotalPorchSF']

# Boolean indicator for fireplace presence
df['HasFireplace'] = df['Fireplaces'].apply(lambda x: 1 if x > 0 else 0).astype('category')

In [ ]:
# Mapping for OverallQual to match ExterQual and KitchenQual ratings
overall_qual_map = {
    10: 'Ex',  # Very Excellent
    9:  'Ex',  # Excellent
    8:  'Gd',  # Very Good
    7:  'Gd',  # Good
    6:  'TA',  # Above Average
    5:  'TA',  # Average
    4:  'Fa',  # Below Average
    3:  'Fa',  # Fair
    2:  'Po',  # Poor
    1:  'Po'   # Very Poor
}

# Apply the mapping to convert OverallQual to qualitative categories
df['OverallQualCategory'] = df['Overall_Qual'].map(overall_qual_map).astype('category')

# Print the updated dataframe to check the new category column
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())


In [ ]:
df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual','Kitchen_Qual']].dtypes

In [ ]:
categorical_features_new = ['AgeGroup', 'HasPool','Remodeled', 'HasFireplace', 'TotalRooms', 'TotalParking', 'TotalBathrooms', 'OverallQualCategory']

**Change data types of newly created `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [ ]:
# for col in categorical_features_new:
#     df[col] = df[col].astype('category')

In [ ]:
# Mapping for ExterQual and KitchenQual
quality_map = {
    'Ex': 5,  # Excellent
    'Gd': 4,  # Good
    'TA': 3,  # Typical/Average
    'Fa': 2,  # Fair
    'Po': 1   # Poor
}

df['Exter_Qual'] = df['Exter_Qual'].map(quality_map)
df['Kitchen_Qual'] = df['Kitchen_Qual'].map(quality_map)
df['OverallQualCategory'] = df['OverallQualCategory'].map(quality_map)

In [ ]:
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())

In [ ]:
print(df[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].dtypes)

In [ ]:
len(df.columns.to_list())

In [ ]:
df.head()

In [ ]:
# Combined quality score
df['Quality_Score'] = ((df['OverallQualCategory'].astype(int) + df['Exter_Qual'].astype(int) + df['Kitchen_Qual'].astype(int)) / 3)

In [ ]:
df.columns.to_list()

In [ ]:
categorical_features_new

In [ ]:
df[categorical_features_new].dtypes

In [ ]:
len(df.columns.to_list())

## 5. Mathematical Transformation

## 6. Categorical Encoding

In [ ]:
categorical_features_new

In [ ]:
categorical_features_old = categorical_features_all

In [ ]:
categorical_features_final = categorical_features_old + categorical_features_new

In [ ]:
print(len(categorical_features_old))
print(len(categorical_features_new))
print(len(categorical_features_final))

In [ ]:
categorical_features_final

In [ ]:
for col in categorical_features_final:
  print(f"{col}: {df[col].nunique()}")

In [ ]:
from sklearn.model_selection import train_test_split, LeaveOneOut, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, scale, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
binary_columns = [col for col in df.columns if (df[col].dtypes not in ["float64","int64","int32"]) & (df[col].nunique() == 2)]
binary_columns

In [ ]:
labelencoder = LabelEncoder()

for col in binary_columns:
    df[col] = labelencoder.fit_transform(df[col])

In [ ]:
ohe_cols = [col for col in df.columns if (df[col].dtype not in ['float64','int64','int32']) & ((df[col].nunique() > 2) | (df[col].nunique() == 1))]
ohe_cols

In [ ]:
df = pd.get_dummies(df, columns=ohe_cols).reset_index(drop=True)
df.shape

In [ ]:
df = df.loc[:,~df.columns.duplicated()]

In [ ]:
len(df.columns.to_list())

> From $96$ to $427$ columns after encoding

## 7. Scaling

In [ ]:
scaling_features=[feature for feature in df.columns if feature not in ['SalePrice'] ]
len(scaling_features)

In [ ]:
df[scaling_features].head()

In [ ]:
minmax_scaler = MinMaxScaler()
df[scaling_features] = minmax_scaler.fit_transform(df[scaling_features])

In [ ]:
numerical_features_all

In [ ]:
set(temporal_features) - set(numerical_features_all)

In [ ]:
len(numerical_features_all)

In [ ]:
numerical_features_new = list(set())

In [ ]:
df['SalePrice']

In [ ]:
df.head()

## 8. `Pipeline` and `ColumnTransformer`

While a `ColumnTransformer` is typically used earlier in the process to apply different transformations to different subsets of columns (e.g., numerical vs. categorical), our data (`df`) has already undergone comprehensive preprocessing, including handling missing values, scaling numerical features, and one-hot encoding categorical features.

At this stage, all features in our `df` are numerical and scaled, making a `ColumnTransformer` redundant for *further* preprocessing steps. However, a `Pipeline` is still invaluable for chaining model steps and ensuring consistency.

I will now re-split our preprocessed `df` into `X` (features) and `y` (target variable `SalePrice`), then define a simple `Pipeline` that includes a model, ready for training.

> On an other notebook

## 9. Modeling (Training)

### Models
**Baseline and Linear Models**
- Linear Regression (OLS)
- Ridge and Lasso Regression

**Tree-Based and Ensemble Models**
- Random Forest Regressor
- XGBoost (Extreme Gradient Boosting)
- LightGBM
- CatBoost

**Non-Linear Models**
- Support Vector Regressor (SVR)
### Evaluation Metrics
- **MAE (Mean Absolute Error)**: Measures average absolute distance between predictions and true values in original currency units.
- **RMSE (Root Mean Squared Error)**: Penalizes larger errors more heavily by squaring differences before taking the square root.
- **R² Score**: Represents the proportion of variance in house prices explained by the model features

In [ ]:
df.tail()

In [ ]:
train_df = df.iloc[:1460]
test_df = df.iloc[1460:]

In [ ]:
# Reset the index for both dataframes
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
#drop Target Feature from test_df
test_df = test_df.drop('SalePrice', axis=1)

In [ ]:
train_df.shape, test_df.shape

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
# Prepare the data
y = df['SalePrice']
X = df.drop(['SalePrice'], axis=1)

In [ ]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
y_train.head()

In [ ]:
y_test.head()

In [ ]:
%pip install catboost

**Import models**

In [ ]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from catboost import CatBoostRegressor

In [ ]:
def evaluate_metrics(true, predicted):
    mse = mean_squared_error(true, predicted) # Calculate Accuracy
    mae = mean_absolute_error(true, predicted) # Calculated MAE
    rmse = root_mean_squared_error(true, predicted) # Calculated RMSE
    r2 = r2_score(true, predicted) # Calculated R2 Score
    return mse, mae, rmse, r2

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(),
    "Ridge Regression": Ridge(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False, random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "Support Vector Regressor": SVR()
}

In [ ]:
# Create a function which can evaluate models and return a report
def evaluate_regressor_models(X_train, y_train, X_test, y_test, models):
    '''
    This function takes in X_train, y_train, X_test, y_test and models dictionary as input
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics with cost
    '''
    models_list = []
    mse_list = []

    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Training set performance
        mse_train, mae_train, rmse_train, r2_train = evaluate_metrics(y_train ,y_train_pred)


        # Test set performance
        mse_test, mae_test, rmse_test, r2_test = evaluate_metrics(y_test, y_test_pred)

        print(list(models.keys())[i])
        models_list.append(list(models.keys())[i])

        print('Model performance for Training set')
        print(f"- MSE: {mse_train:.4f}")
        print(f'- MAE: {mae_train:.4f}')
        print(f'- RMSE: {rmse_train:.4f}')
        print(f'- R2: {r2_train:.4f}')

        print('----------------------------------')

        print('Model performance for Test set')
        print(f'- MSE: {mse_test:.4f}')
        mse_list.append(mse_test)
        print(f'- MAE: {mae_test:.4f}')
        print(f'- RMSE: {rmse_test:.4f}')
        print(f'- R2: {r2_test:.4f}')
        print('='*35)
        print('\n')

    report=pd.DataFrame(list(zip(models_list, mse_list)), columns=['Model Name', 'MSE']).sort_values(by=['MSE'], ascending=True) # Sort by MSE ascending, lower is better

    return report

In [ ]:
base_model_report = evaluate_regressor_models(X_train, y_train, X_test, y_test, models)

**Model Performance Visualization**

In [ ]:
print('\nModel Performance Visualization')
for model_name, regressor in models.items():
    # Ensure the model is trained before predicting
    # The evaluate_regressor_models function already trains the models
    # So, we can directly predict using the models in the 'models' dictionary.
    y_pred = regressor.predict(X_test)

    plt.figure(figsize=(16, 6))

    # Regression Plot
    plt.subplot(1, 2, 1)
    sns.regplot(x=y_pred, y=y_test, scatter_kws={'alpha':0.4})
    plt.xlabel('Predicted Price')
    plt.ylabel('Actual Price')
    plt.title(f'Regression Plot: Actual vs Predicted ({model_name})')

    # Residual Plot
    plt.subplot(1, 2, 2)
    residuals = y_test - y_pred
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.4)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Price')
    plt.ylabel('Residuals')
    plt.title(f'Residual Plot ({model_name})')

    plt.tight_layout()
    plt.show()

**Cross Validation**

In [ ]:
from sklearn.model_selection import cross_validate

print("Cross-Validation Results:")
for model_name, regressor in models.items():
    scoring = {'mae': 'neg_mean_absolute_error', 'mse': 'neg_mean_squared_error', 'r2': 'r2'}
    cv_results = cross_validate(regressor, X, y, cv=5, scoring=scoring)
    print(f"\n{model_name}:")
    print(f"  Average MAE: {-cv_results['test_mae'].mean():.4f}")
    print(f"  Average MSE: {-cv_results['test_mse'].mean():.4f}")
    print(f"  Average RMSE: {np.sqrt(-cv_results['test_mse'].mean()):.4f}")
    print(f"  Average R2: {cv_results['test_r2'].mean():.4f}")

**Hyperparameter Tuning**

In [ ]:
# default parameters
print("Default Parameters:")
for model_name, regressor in models.items():
    print(f'{model_name}')
    print(f'parameters: {regressor.get_params()}', end='\n\n')

In [ ]:
# Define parameter grids for hyperparameter tuning
model_params = [
    ('Linear Regression', LinearRegression(), {
        'fit_intercept': [True, False]
    }),
    ('Lasso Regression', Lasso(), {
        'alpha': [0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }),
    ('Ridge Regression', Ridge(), {
        'alpha': [0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }),
    ('Random Forest', RandomForestRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }),
    ('GBM', GradientBoostingRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'max_depth': [3, 5, 7]
    })
]

In [ ]:
# Best parameters and scores initialization
best_model_name = None
best_model = None
best_score = float('-inf')
best_params = None

print("\nHyperparameter Tuning Results:")
for model_name, regressor, params in model_params:
    regressor_grid = GridSearchCV(regressor, params, cv=5, n_jobs=-1, verbose=1).fit(X, y)

    print(f"\nModel: {model_name}")
    print("Best Parameters:", regressor_grid.best_params_)
    print("Best Score:", regressor_grid.best_score_)
    print("\n")

    if regressor_grid.best_score_ > best_score:
        best_model_name = model_name
        best_model = regressor
        best_score = regressor_grid.best_score_
        best_params = regressor_grid.best_params_

print("\n\nFinal Inference")
print(f"Model: {best_model_name}")
print("Best Parameters:", best_params)
print("Best Score:", best_score)


In [ ]:
best_model

In [ ]:
# Function to find best model parameters
def best_params(model_params, best_model):
    for model_name, regressor, params in model_params.items():
        if best_model == regressor:
            return params
    return None

In [ ]:
# Apply the function
best_model_params = best_params(model_params, best_model)

In [ ]:
best_model_params

In [ ]:
# Final GridSearchCV with best model and its parameter grid
regressor_best_grid = GridSearchCV(best_model, best_model_params, cv=5, n_jobs=-1, verbose=1).fit(X, y)

In [ ]:
regressor_best_grid

In [ ]:
final_model = GradientBoostingRegressor(**regressor_best_grid.best_params_).fit(X, y)

**Cross Validation**

In [ ]:
print("Best Model Cross-Validation")
scoring = {'mae': 'neg_mean_absolute_error', 'mse': 'neg_mean_squared_error', 'r2': 'r2'}
cv_results_final = cross_validate(final_model, X, y, cv=5, scoring=scoring)
print(f"\n{model_name}:")
print(f"  Average MAE: {-cv_results['test_mae'].mean():.4f}")
print(f"  Average MSE: {-cv_results['test_mse'].mean():.4f}")
print(f"  Average RMSE: {np.sqrt(-cv_results['test_mse'].mean()):.4f}")
print(f"  Average R2: {cv_results['test_r2'].mean():.4f}", end="\n\n")

**Feature Importance**

In [ ]:
# creating function to visualize:

def visualize_importance(model, features, start=0, num=len(X), save=False):
    feature_imp = pd.DataFrame({'Value': model.feature_importances_, 'Feature': features.columns})
    sorted_feature_imp = feature_imp.sort_values("Value", ascending=False)[start:num]
    display(sorted_feature_imp)
    #print(feature_imp.sort_values("Value",ascending=False)[start:num])
    plt.figure(figsize=(12, 12))
    sns.set(font_scale=0.7)
    palette = sns.color_palette("coolwarm", n_colors=num-start)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",ascending=False)[start:num], palette=palette)
    plt.title('Visualization of Important Features')
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_importance(final_model, X,start=0, num=45)

In [ ]:
predictions = final_model.predict(test_df.drop(['Id'], axis=1))

In [ ]:
predictions

In [ ]:
test_sample = test_df.drop(['Id'], axis=1).iloc[0].values.reshape(1, -1)
prediction = final_model.predict(test_sample)
print(prediction)

In [ ]:
predict_df = test_df.copy()
predict_df = predict_df.drop(['Id'], axis=1)

## Saving model with Pickle

In [ ]:
import pickle

with open('model/ames_ml_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

print("Model saved to ames_ml_model.pkl")

**Loading model and predicting using it**

In [ ]:
# Load model from pickle file
with open('model/ames_ml_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

result = loaded_model.score(X_test, y_test)
print(result)

print('Predicted SalePrice is : ', loaded_model.predict(predict_df))